# Proyek Akhir Praktikum Big Data: TidarMart Data Platform
**Nama:** Wildan Maulana Habibi  
**NPM:** 2505060022  
**Seed:** 60022  
**Varian Pertanyaan Bisnis:** 2 (Fokus Produk dan Diskon)  

---

## Tahap 1: Penyimpanan Terdistribusi dan Ingesti (HDFS)

### Tujuan
1. Menyiapkan struktur direktori penyimpanan terdistribusi pada HDFS (`/tidarmart/raw`, `/tidarmart/stream_input`, `/tidarmart/curated`, dan `/tidarmart/models`).
2. Mengunggah seluruh hasil generator dataset (`transaksi`, `pelanggan.json`, `produk.json`, dan `stream_input`) dari folder lokal `data_proyek/` ke HDFS.
3. Memverifikasi ukuran berkas, jumlah blok penyimpanan, serta faktor replikasi di HDFS (`hdfs dfs -du -h` dan `hdfs fsck`).
4. Memastikan PySpark dapat membaca seluruh dataset langsung dari protokol `hdfs://localhost:9000/...` sesuai batasan memori VM.

In [2]:
#Buat struktur direktori di HDFS
!hdfs dfs -mkdir -p /tidarmart/raw/transaksi
!hdfs dfs -mkdir -p /tidarmart/raw/pelanggan
!hdfs dfs -mkdir -p /tidarmart/raw/produk
!hdfs dfs -mkdir -p /tidarmart/stream_input
!hdfs dfs -mkdir -p /tidarmart/curated
!hdfs dfs -mkdir -p /tidarmart/models

#Unggah dataset dari ./data_proyek/ ke masing-masing folder HDFS
!hdfs dfs -put -f data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
!hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
!hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk/
!hdfs dfs -put -f data_proyek/stream_input/*.csv /tidarmart/stream_input/

print("Seluruh dataset berhasil diunggah ke HDFS.")

Seluruh dataset berhasil diunggah ke HDFS.


In [3]:
print("=== UKURAN BERKAS DI HDFS ===")
!hdfs dfs -du -h /tidarmart/raw
!hdfs dfs -du -h /tidarmart/stream_input

print("\n=== JUMLAH BLOK & FAKTOR REPLIKASI ===")
!hdfs fsck /tidarmart/raw -files -blocks

=== UKURAN BERKAS DI HDFS ===
6.0 M    6.0 M    /tidarmart/raw/pelanggan
680.7 K  680.7 K  /tidarmart/raw/produk
187.2 M  187.2 M  /tidarmart/raw/transaksi
479.7 K  479.7 K  /tidarmart/stream_input/stream_01.csv
479.7 K  479.7 K  /tidarmart/stream_input/stream_02.csv
479.3 K  479.3 K  /tidarmart/stream_input/stream_03.csv
479.1 K  479.1 K  /tidarmart/stream_input/stream_04.csv
479.7 K  479.7 K  /tidarmart/stream_input/stream_05.csv
479.6 K  479.6 K  /tidarmart/stream_input/stream_06.csv
479.2 K  479.2 K  /tidarmart/stream_input/stream_07.csv
478.9 K  478.9 K  /tidarmart/stream_input/stream_08.csv
479.4 K  479.4 K  /tidarmart/stream_input/stream_09.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_10.csv
479.4 K  479.4 K  /tidarmart/stream_input/stream_11.csv
479.6 K  479.6 K  /tidarmart/stream_input/stream_12.csv
479.8 K  479.8 K  /tidarmart/stream_input/stream_13.csv
479.2 K  479.2 K  /tidarmart/stream_input/stream_14.csv
479.8 K  479.8 K  /tidarmart/stream_input/stream_15.csv
479.

In [3]:
from pyspark.sql import SparkSession

# Inisialisasi SparkSession 
spark = (SparkSession.builder
    .appName('TidarMart')
    .master('local[2]')
    .config('spark.driver.memory', '1536m')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())

# Membaca seluruh data dari HDFS 
df_transaksi = spark.read.csv("hdfs://localhost:9000/tidarmart/raw/transaksi/*.csv", header=True)
df_pelanggan = spark.read.json("hdfs://localhost:9000/tidarmart/raw/pelanggan/pelanggan.json")
df_produk    = spark.read.json("hdfs://localhost:9000/tidarmart/raw/produk/produk.json")

# Verifikasi jumlah baris
print(f"Transaksi terbaca : {df_transaksi.count():,} baris")
print(f"Pelanggan terbaca : {df_pelanggan.count():,} dokumen")
print(f"Produk terbaca    : {df_produk.count():,} dokumen")

26/11/26 11:18:24 WARN Utils: Your hostname, xicrth resolves to a loopback address: 127.0.1.1; using 10.90.69.179 instead (on interface wlp62s0)
26/11/26 11:18:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 11:18:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Transaksi terbaca : 2,010,000 baris
Pelanggan terbaca : 50,000 dokumen
Produk terbaca    : 5,000 dokumen


### Jawaban Pertanyaan Konseptual & Interpretasi Tahap 1

**1. Mengapa direktori data mentah (`raw`) dan data bersih (`curated`) dipisah?**
* **Menjaga Keaslian Data (*Immutability*):** Direktori `/tidarmart/raw` berfungsi sebagai sumber kebenaran asli (*single source of truth*) dari hasil ingesti[cite: 4]. Apabila terjadi kesalahan logika saat proses pembersihan (ETL) atau perubahan aturan bisnis di kemudian hari, proses pembersihan dapat diulang dari awal tanpa merusak atau kehilangan data mentah[cite: 4].
* **Efisiensi Komputasi dan Keamanan Akses:** Direktori `/tidarmart/curated` menyimpan data yang sudah bersih dan terformat dalam struktur kolumnar (Parquet)[cite: 4]. Tahap analitik maupun *machine learning* cukup membaca dari `curated` sehingga eksekusi lebih cepat dan hemat memori dibandingkan memproses ulang data kotor[cite: 2, 4].

**2. Apa arti faktor replikasi $= 1$ pada VM *pseudo-distributed* dibanding klaster nyata?**
* **Pada VM *Pseudo-Distributed*:** Seluruh layanan HDFS (*NameNode* dan *DataNode*) berjalan pada satu mesin virtual dan satu disk fisik yang sama[cite: 1, 4]. Mengatur faktor replikasi lebih dari 1 tidak memberikan perlindungan terhadap kegagalan perangkat keras, melainkan hanya menggandakan pemakaian ruang penyimpanan VM dan memperlambat proses tulis (*disk I/O*)[cite: 3, 4].
* **Pada Klaster Produksi Nyata (*Multi-Node*):** Faktor replikasi standarnya diatur menjadi **3** (setiap blok data disalin ke tiga server/rak yang berbeda)[cite: 4]. Hal ini menjamin **toleransi kesalahan (*fault tolerance*)** agar data tidak hilang jika ada server yang mati, serta mempercepat pembacaan paralel melalui prinsip **lokalitas data (*data locality*)**.

**Hasil Verifikasi:**  
Data yang diunggah ke HDFS telah berhasil dibaca oleh PySpark melalui jalur `hdfs://localhost:9000/tidarmart/...` dengan jumlah baris yang cocok sepenuhnya dengan keluaran generator (**2.010.000** baris transaksi, **50.000** pelanggan, dan **5.000** produk)[cite: 4, 8].

## Tahap 2: Profiling Kualitas Data (Veracity)

### Tujuan
1. Mendefinisikan skema eksplisit (`StructType`) untuk membaca sumber JSON (`pelanggan.json` dan `produk.json`) serta mengevaluasi opsi pembacaan CSV transaksi.
2. Mengidentifikasi seluruh permasalahan kualitas data (*Veracity*) pada dataset transaksi: duplikat `order_id`, persentase nilai kosong (*null*), harga negatif, outlier pada `qty`, dan inkonsistensi penulisan nama `kota`.
3. Menyajikan hasil profiling dalam satu tabel ringkasan **"Sebelum Pembersihan"**.
4. Memeriksa struktur semi-terstruktur pada `pelanggan.json` dan `produk.json` (kelengkapan atribut khusus per kategori dan perlakuan terhadap field array `tags`).

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
import pandas as pd

#Skema eksplisit untuk JSON
schema_pelanggan = StructType([
    StructField("pelanggan_id", IntegerType(), True),
    StructField("nama", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("tanggal_daftar", StringType(), True),
    StructField("membership", StringType(), True)
])

schema_produk = StructType([
    StructField("produk_id", IntegerType(), True),
    StructField("nama_produk", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("harga_dasar", IntegerType(), True),
    StructField("berat_kg", DoubleType(), True),
    StructField("garansi_bulan", IntegerType(), True),
    StructField("voltase", IntegerType(), True),
    StructField("ukuran", StringType(), True),
    StructField("bahan", StringType(), True),
    StructField("kadaluarsa_hari", IntegerType(), True),
    StructField("penerbit", StringType(), True),
    StructField("tags", ArrayType(StringType()), True)
])

# Membaca data dari HDFS
df_trx = spark.read.csv("hdfs://localhost:9000/tidarmart/raw/transaksi/*.csv", header=True, inferSchema=True)
df_pel = spark.read.schema(schema_pelanggan).json("hdfs://localhost:9000/tidarmart/raw/pelanggan/pelanggan.json")
df_prd = spark.read.schema(schema_produk).json("hdfs://localhost:9000/tidarmart/raw/produk/produk.json")

# Profiling Jumlah Baris, Duplikat, Harga Negatif, dan Outlier Kuantil
total_baris = df_trx.count()
unik_order  = df_trx.select("order_id").distinct().count()
duplikat    = total_baris - unik_order
harga_neg   = df_trx.filter(F.col("harga_satuan") < 0).count()

# Hitung kuantil qty (Q1, Median, Q3, P99, Max)
kuantil_qty = df_trx.approxQuantile("qty", [0.25, 0.50, 0.75, 0.99, 1.0], 0.01)
q1, med, q3, p99, max_qty = kuantil_qty
iqr = q3 - q1
batas_atas_iqr = q3 + 1.5 * iqr
outlier_qty = df_trx.filter(F.col("qty") > batas_atas_iqr).count()

print(f"Distribusi qty -> Q1: {q1}, Median: {med}, Q3: {q3}, P99: {p99}, Max: {max_qty}")
print(f"Batas atas IQR (Q3 + 1.5*IQR): {batas_atas_iqr} | Jumlah outlier qty: {outlier_qty:,}")

Distribusi qty -> Q1: 2.0, Median: 4.0, Q3: 6.0, P99: 999.0, Max: 999.0
Batas atas IQR (Q3 + 1.5*IQR): 12.0 | Jumlah outlier qty: 1,041


In [5]:
# Persentase nilai kosong (null) per kolom
print("=== PERSENTASE NILAI KOSONG (NULL) PER KOLOM ===")
null_counts = df_trx.select([
    F.round((F.count(F.when(F.col(c).isNull(), c)) / total_baris) * 100, 3).alias(c)
    for c in df_trx.columns
])
null_counts.show(vertical=True)

# Variasi penulisan nama kota
print("=== VARIASI PENULISAN NAMA KOTA ===")
df_trx.groupBy("kota").count().orderBy(F.desc("count")).show(30, truncate=False)

# Tabel Ringkasan "Sebelum Pembersihan"
null_qty = df_trx.filter(F.col("qty").isNull()).count()
null_hrg = df_trx.filter(F.col("harga_satuan").isNull()).count()
kota_tidak_standar = df_trx.filter(F.col("kota") != F.initcap(F.col("kota"))).count()

ringkasan_sebelum = pd.DataFrame([
    {"Metrik Kualitas Data": "Total Baris Data", "Nilai": f"{total_baris:,}", "Persentase": "100.00%"},
    {"Metrik Kualitas Data": "Jumlah order_id Unik", "Nilai": f"{unik_order:,}", "Persentase": f"{(unik_order/total_baris)*100:.2f}%"},
    {"Metrik Kualitas Data": "Baris Duplikat (order_id ganda)", "Nilai": f"{duplikat:,}", "Persentase": f"{(duplikat/total_baris)*100:.2f}%"},
    {"Metrik Kualitas Data": "Nilai Kosong pada kolom qty", "Nilai": f"{null_qty:,}", "Persentase": f"{(null_qty/total_baris)*100:.2f}%"},
    {"Metrik Kualitas Data": "Nilai Kosong pada kolom harga_satuan", "Nilai": f"{null_hrg:,}", "Persentase": f"{(null_hrg/total_baris)*100:.2f}%"},
    {"Metrik Kualitas Data": "Harga Satuan Negatif (< 0)", "Nilai": f"{harga_neg:,}", "Persentase": f"{(harga_neg/total_baris)*100:.2f}%"},
    {"Metrik Kualitas Data": f"Outlier Kuantitas (qty > {batas_atas_iqr})", "Nilai": f"{outlier_qty:,}", "Persentase": f"{(outlier_qty/total_baris)*100:.2f}%"},
    {"Metrik Kualitas Data": "Penulisan Kota Tidak Standar (huruf besar/kecil)", "Nilai": f"{kota_tidak_standar:,}", "Persentase": f"{(kota_tidak_standar/total_baris)*100:.2f}%"}
])

print("=== TABEL RINGKASAN SEBELUM PEMBERSIHAN ===")
display(ringkasan_sebelum)

=== PERSENTASE NILAI KOSONG (NULL) PER KOLOM ===


-RECORD 0------------------
 order_id          | 0.0   
 tanggal           | 0.0   
 pelanggan_id      | 0.0   
 produk_id         | 0.0   
 qty               | 0.294 
 harga_satuan      | 0.304 
 diskon_persen     | 0.0   
 metode_pembayaran | 0.0   
 kota              | 0.0   
 berat_kg          | 0.0   
 jarak_km          | 0.0   
 lama_kirim_hari   | 0.0   
 status_pesanan    | 0.0   

=== VARIASI PENULISAN NAMA KOTA ===


+----------+------+
|kota      |count |
+----------+------+
|Magelang  |239202|
|Yogyakarta|239045|
|Purworejo |238799|
|Kebumen   |238761|
|Temanggung|238650|
|Solo      |238631|
|Semarang  |238412|
|Wonosobo  |238058|
|kebumen   |6457  |
|WONOSOBO  |6397  |
|MAGELANG  |6345  |
|SOLO      |6342  |
|SEMARANG  |6314  |
|semarang  |6283  |
|yogyakarta|6282  |
|PURWOREJO |6279  |
|temanggung|6276  |
|TEMANGGUNG|6237  |
|purworejo |6231  |
|YOGYAKARTA|6230  |
|KEBUMEN   |6214  |
|solo      |6204  |
|magelang  |6189  |
|wonosobo  |6162  |
+----------+------+



=== TABEL RINGKASAN SEBELUM PEMBERSIHAN ===


,Metrik Kualitas Data,Nilai,Persentase
0,Total Baris Data,"2,010,000",100.00%
1,Jumlah order_id Unik,"2,000,000",99.50%
2,Baris Duplikat (order_id ganda),"10,000",0.50%
3,Nilai Kosong pada kolom qty,"5,917",0.29%
4,Nilai Kosong pada kolom harga_satuan,"6,105",0.30%
5,Harga Satuan Negatif (< 0),"2,012",0.10%
6,Outlier Kuantitas (qty > 12.0),"1,041",0.05%
7,Penulisan Kota Tidak Standar (huruf besar/kecil),"100,442",5.00%


In [6]:
# Profil Pelanggan
print("=== PROFIL PELANGGAN ===")
print(f"Total pelanggan: {df_pel.count():,} | Pelanggan ID unik: {df_pel.select('pelanggan_id').distinct().count():,}")
df_pel.groupBy("membership").count().show()

# Profil Produk: Persentase kemunculan atribut khusus per kategori & field tags
print("=== PERSENTASE KELENGKAPAN ATRIBUT PRODUK PER KATEGORI ===")
atribut_khusus = ["garansi_bulan", "voltase", "ukuran", "bahan", "kadaluarsa_hari", "penerbit", "tags"]

df_profil_produk = df_prd.groupBy("kategori").agg(
    F.count("*").alias("jml_produk"),
    *[F.round((F.count(F.col(attr)) / F.count("*")) * 100, 1).alias(f"{attr}_%") for attr in atribut_khusus]
).orderBy("kategori")

df_profil_produk.show(truncate=False)

# Contoh produk dengan field array tags vs tanpa tags
print("=== CONTOH FIELD ARRAY TAGS PADA PRODUK ===")
df_prd.select("produk_id", "kategori", "tags").show(10, truncate=False)

=== PROFIL PELANGGAN ===
Total pelanggan: 50,000 | Pelanggan ID unik: 50,000
+----------+-----+
|membership|count|
+----------+-----+
|      Gold|15000|
|  Platinum| 4946|
|    Silver|30054|
+----------+-----+

=== PERSENTASE KELENGKAPAN ATRIBUT PRODUK PER KATEGORI ===
+------------+----------+---------------+---------+--------+-------+-----------------+----------+------+
|kategori    |jml_produk|garansi_bulan_%|voltase_%|ukuran_%|bahan_%|kadaluarsa_hari_%|penerbit_%|tags_%|
+------------+----------+---------------+---------+--------+-------+-----------------+----------+------+
|Buku        |685       |0.0            |0.0      |0.0     |0.0    |0.0              |100.0     |30.7  |
|Elektronik  |562       |100.0          |100.0    |0.0     |0.0    |0.0              |0.0       |29.9  |
|Fashion     |622       |0.0            |0.0      |100.0   |100.0  |0.0              |0.0       |33.0  |
|Kesehatan   |632       |0.0            |0.0      |0.0     |0.0    |0.0              |0.0       |26.

### Analisis Hasil Profiling Tahap 2

**1. Trade-off Penggunaan `inferSchema=True` pada CSV vs Skema Eksplisit (`StructType`)**
* Pada berkas CSV transaksi, `inferSchema=True` mempermudah deteksi tipe data otomatis, namun memiliki *trade-off* berupa **pembacaan data sebanyak dua kali (*two-pass*)** (lintasan pertama untuk menebak tipe data tiap kolom, lintasan kedua untuk memuat data), yang menambah waktu dan beban I/O disk[cite: 3].
* Sebaliknya, pada berkas JSON (`pelanggan.json` dan `produk.json`), digunakan **skema eksplisit (`StructType`)** karena struktur atribut produk bersifat semi-terstruktur dan tidak seragam antar kategori[cite: 1, 3]. Skema eksplisit memastikan semua kemungkinan kolom langsung terpetakan dalam satu kali baca (*single-pass*) tanpa risiko tipe data salah tebak[cite: 3].

**2. Temuan Kualitas Data (*Veracity*) pada Transaksi**
* **Duplikasi:** Ditemukan ±0,5% baris duplikat dengan `order_id` yang sama persis akibat proses pencatatan ulang[cite: 3, 8].
* **Nilai Kosong (*Null*):** Hanya kolom `qty` dan `harga_satuan` yang memiliki nilai kosong (masing-masing ±0,3%), sedangkan kolom lain terisi penuh[cite: 3, 8].
* **Harga Negatif:** Terdapat ±0,1% transaksi dengan `harga_satuan` bernilai negatif (kesalahan tanda minus saat input)[cite: 3, 8].
* **Outlier Kuantitas:** Berdasarkan perhitungan kuantil ($Q_1$, $Q_3$, dan $IQR$), mayoritas pesanan memiliki `qty` antara 1 hingga 7 unit, namun terdapat ±0,05% anomali ekstrem bernilai `999`[cite: 3, 8].
* **Inkonsistensi Penulisan Kota:** Sekitar 5% nama kota ditulis seluruhnya huruf kapital (`MAGELANG`, `SOLO`) atau huruf kecil semua (`magelang`, `solo`), yang akan memecah hasil pengelompokan jika tidak dinormalisasi[cite: 3, 8].

**3. Struktur Semi-Terstruktur pada Produk & Perlakuan Field `tags`**
* Hasil agregasi menunjukkan bahwa atribut khusus hanya muncul 100% pada kategori yang relevan (`garansi_bulan` dan `voltase` hanya pada *Elektronik*; `ukuran` dan `bahan` pada *Fashion*; `kadaluarsa_hari` pada *Makanan*; `penerbit` pada *Buku*), sementara kategori lain bernilai `null` (0%)[cite: 8].
* Field `tags` merupakan tipe **array opsional** yang hanya dimiliki oleh sekitar 30% produk[cite: 1, 8]. Di PySpark, field ini didefinisikan sebagai `ArrayType(StringType())` sehingga produk yang tidak memiliki tag tetap aman terbaca sebagai `null` tanpa menyebabkan error parsing, dan dapat dipecah menggunakan fungsi `explode()` saat dibutuhkan analisis per tag[cite: 3, 8].